# 08 · SHAP Explainability

**Project:** Enterprise HR AI  
**Model:** `logistic_regression_balanced` from `models/attrition_pipeline.joblib`  
**Explainer:** `shap.LinearExplainer` — the correct choice for Logistic Regression.
TreeExplainer and KernelExplainer are wrong for this model type and are NOT used here.

---

In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')          # non-interactive backend — required for saving PNGs
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split

PROC    = os.path.join('..', 'data', 'processed')
MODELS  = os.path.join('..', 'models')
REPORTS = os.path.join('..', 'reports', 'shap')
os.makedirs(REPORTS, exist_ok=True)

RANDOM_STATE = 42
print('shap version  :', shap.__version__)
print('REPORTS dir   :', os.path.abspath(REPORTS))

shap version  : 0.52.0
REPORTS dir   : C:\Users\ASUS\Desktop\enterprise_hr_ai\reports\shap


---
## Step 1 · Load Model & Config

Confirm the loaded model matches `model_config.json` (logistic_regression_balanced, threshold=0.40).

In [2]:
# Load config
with open(os.path.join(MODELS, 'model_config.json'), 'r') as f:
    config = json.load(f)
print('=== model_config.json ===')
print(json.dumps(config, indent=2))

# Load model
model = joblib.load(os.path.join(MODELS, 'attrition_pipeline.joblib'))
print()
print('Loaded model type :', type(model).__name__)
print('class_weight      :', model.class_weight)
print('max_iter          :', model.max_iter)
print()
# Confirm model type matches config
assert config['model'] == 'logistic_regression_balanced', \
    f'Config model mismatch: {config["model"]}'
assert config['threshold'] == 0.40, \
    f'Config threshold mismatch: {config["threshold"]}'
print('CONFIRMED: model=logistic_regression_balanced, threshold=0.40 — matches config.')

=== model_config.json ===
{
  "model": "logistic_regression_balanced",
  "threshold": 0.4,
  "recall": 0.7872,
  "precision": 0.3426,
  "f1": 0.4774,
  "trained_on": "features_scaled.csv",
  "date": "2026-09-01"
}

Loaded model type : LogisticRegression
class_weight      : balanced
max_iter          : 1000

CONFIRMED: model=logistic_regression_balanced, threshold=0.40 — matches config.


---
## Step 2 · Recreate Identical 80/20 Stratified Split

In [3]:
df = pd.read_csv(os.path.join(PROC, 'features_scaled.csv'))
X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Test attrition rate: {y_test.mean()*100:.2f}%  (15.99% expected)')
assert abs(y_test.mean()*100 - 15.99) < 0.05, 'Split mismatch!'
print('CONFIRMED: Split identical to notebooks 06/07.')

Train: (1176, 48), Test: (294, 48)
Test attrition rate: 15.99%  (15.99% expected)
CONFIRMED: Split identical to notebooks 06/07.


---
## Step 3 · SHAP LinearExplainer

**Why LinearExplainer?**  
For a linear model (Logistic Regression), SHAP values decompose as:
`phi_j = coef_j * (x_j - E[x_j])`.  
LinearExplainer computes this exactly in O(n·p) time.  
TreeExplainer requires a tree model; KernelExplainer is a model-agnostic
approximation that would be 100–1000× slower with no accuracy benefit here.

In [4]:
print('Initialising shap.LinearExplainer on training set background...')
explainer = shap.LinearExplainer(model, X_train, feature_perturbation='interventional')
print('Explainer type     :', type(explainer).__name__)
print('Expected value     :', explainer.expected_value)   # baseline log-odds
print()

# Compute SHAP values on the test set (Explanation object — new API)
explanation = explainer(X_test)
print('explanation.values shape:', explanation.values.shape)   # (294, 48)
print('Positive class (left=1) SHAP values — correct slice used automatically by LinearExplainer')

Background dataset has 1176 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1176 when initializing the masker.


Initialising shap.LinearExplainer on training set background...
Explainer type     : LinearExplainer
Expected value     : -0.7468997000332016

explanation.values shape: (294, 48)
Positive class (left=1) SHAP values — correct slice used automatically by LinearExplainer


---
## Step 4 · Plot 1 — Beeswarm Summary Plot

In [5]:
fig, ax = plt.subplots(figsize=(10, 12))
plt.sca(ax)
shap.plots.beeswarm(explanation, max_display=20, show=False)
plt.title('SHAP Beeswarm — Attrition Drivers (LogReg Balanced)', fontsize=13, pad=12)
plt.tight_layout()
beeswarm_path = os.path.join(REPORTS, 'summary_beeswarm.png')
plt.savefig(beeswarm_path, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'Saved: {os.path.abspath(beeswarm_path)}  ({os.path.getsize(beeswarm_path):,} bytes)')

Saved: C:\Users\ASUS\Desktop\enterprise_hr_ai\reports\shap\summary_beeswarm.png  (204,292 bytes)


---
## Step 5 · Plot 2 — Global Feature Importance Bar Chart

In [6]:
fig, ax = plt.subplots(figsize=(10, 10))
plt.sca(ax)
shap.plots.bar(explanation, max_display=20, show=False)
plt.title('Mean |SHAP value| — Global Feature Importance', fontsize=13, pad=12)
plt.tight_layout()
bar_path = os.path.join(REPORTS, 'global_importance.png')
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'Saved: {os.path.abspath(bar_path)}  ({os.path.getsize(bar_path):,} bytes)')

# Print top-10 numeric ranking
mean_abs_shap = pd.Series(
    np.abs(explanation.values).mean(axis=0),
    index=X.columns
).sort_values(ascending=False)

print()
print('=== TOP 10 FEATURES BY MEAN |SHAP VALUE| ===')
print(f'{"Rank":<5} {"Feature":<40} {"Mean |SHAP|":>12}')
print('-'*60)
for rank, (feat, val) in enumerate(mean_abs_shap.head(10).items(), 1):
    print(f'{rank:<5} {feat:<40} {val:>12.4f}')

Saved: C:\Users\ASUS\Desktop\enterprise_hr_ai\reports\shap\global_importance.png  (169,371 bytes)

=== TOP 10 FEATURES BY MEAN |SHAP VALUE| ===
Rank  Feature                                   Mean |SHAP|
------------------------------------------------------------
1     OverTime                                       0.6558
2     YearsSinceLastPromotion                        0.5663
3     TotalWorkingYears                              0.5573
4     BusinessTravel_Travel_Frequently               0.5177
5     JobLevel                                       0.4585
6     JobRole_Laboratory Technician                  0.4064
7     MaritalStatus_Single                           0.3873
8     BusinessTravel_Travel_Rarely                   0.3645
9     NumCompaniesWorked                             0.3604
10    YearsWithCurrManager                           0.3441


---
## Step 6 · Individual Employee Examples

Selecting:
- **True Positive:** An employee who actually left AND the model (threshold=0.40) predicted they would leave.
- **True Negative:** An employee who actually stayed AND the model predicted they would stay.

Chosen as the highest-confidence example of each type (most extreme predicted probability within its class).

In [7]:
THRESHOLD = config['threshold']   # 0.40

# Predicted probabilities on test set
probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= THRESHOLD).astype(int)

y_arr = y_test.values

# True Positives: actual=1, predicted=1
tp_mask = (y_arr == 1) & (preds == 1)
tp_indices = np.where(tp_mask)[0]
# Pick the one with highest predicted probability (most confident TP)
tp_idx = tp_indices[np.argmax(probs[tp_indices])]

# True Negatives: actual=0, predicted=0
tn_mask = (y_arr == 0) & (preds == 0)
tn_indices = np.where(tn_mask)[0]
# Pick the one with lowest predicted probability (most confident TN)
tn_idx = tn_indices[np.argmin(probs[tn_indices])]

print('=== INDIVIDUAL EXAMPLE SELECTION ===')
print(f'Total TPs in test set: {tp_mask.sum()} (model correctly flagged as leaving at threshold={THRESHOLD})')
print(f'Total TNs in test set: {tn_mask.sum()} (model correctly predicted as staying at threshold={THRESHOLD})')
print()
print(f'Selected TRUE POSITIVE example: test index [{tp_idx}]')
print(f'  Actual label: {y_arr[tp_idx]} (left=1)')
print(f'  Predicted probability: {probs[tp_idx]:.4f}  (threshold={THRESHOLD})')
print(f'  Rationale: highest-confidence true positive — clearest attrition signal in test set')
print()
print(f'Selected TRUE NEGATIVE example: test index [{tn_idx}]')
print(f'  Actual label: {y_arr[tn_idx]} (stayed=0)')
print(f'  Predicted probability: {probs[tn_idx]:.4f}  (threshold={THRESHOLD})')
print(f'  Rationale: lowest-confidence true negative — employee the model is most certain stayed')

=== INDIVIDUAL EXAMPLE SELECTION ===
Total TPs in test set: 37 (model correctly flagged as leaving at threshold=0.4)
Total TNs in test set: 176 (model correctly predicted as staying at threshold=0.4)

Selected TRUE POSITIVE example: test index [214]
  Actual label: 1 (left=1)
  Predicted probability: 0.9899  (threshold=0.4)
  Rationale: highest-confidence true positive — clearest attrition signal in test set

Selected TRUE NEGATIVE example: test index [260]
  Actual label: 0 (stayed=0)
  Predicted probability: 0.0041  (threshold=0.4)
  Rationale: lowest-confidence true negative — employee the model is most certain stayed


---
## Step 7 · Plot 3 & 4 — Waterfall Plots for Individual Employees

In [8]:
# Waterfall: True Positive (leaver)
fig = plt.figure(figsize=(12, 8))
shap.plots.waterfall(explanation[tp_idx], max_display=15, show=False)
plt.title(f'SHAP Waterfall — Correctly Predicted LEAVER (test index {tp_idx})', fontsize=12)
plt.tight_layout()
wf_leaver_path = os.path.join(REPORTS, 'waterfall_leaver_example.png')
plt.savefig(wf_leaver_path, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'Saved: {os.path.abspath(wf_leaver_path)}  ({os.path.getsize(wf_leaver_path):,} bytes)')

# Print top driving SHAP values for this employee
emp_shap = pd.Series(explanation[tp_idx].values, index=X.columns).sort_values(key=abs, ascending=False)
print(f'\nTop 8 SHAP drivers for LEAVER (index {tp_idx}):')
print(f'{"Feature":<40} {"SHAP value":>12}  Direction')
print('-'*65)
for feat, val in emp_shap.head(8).items():
    direction = '-> TOWARD leaving (+)' if val > 0 else '-> TOWARD staying (-)'
    print(f'{feat:<40} {val:>12.4f}  {direction}')

Saved: C:\Users\ASUS\Desktop\enterprise_hr_ai\reports\shap\waterfall_leaver_example.png  (168,520 bytes)

Top 8 SHAP drivers for LEAVER (index 214):
Feature                                    SHAP value  Direction
-----------------------------------------------------------------
BusinessTravel_Travel_Frequently               1.2656  -> TOWARD leaving (+)
JobRole_Sales Representative                   1.1714  -> TOWARD leaving (+)
OverTime                                       1.1380  -> TOWARD leaving (+)
TotalWorkingYears                              0.8045  -> TOWARD leaving (+)
BusinessTravel_Travel_Rarely                  -0.6424  -> TOWARD staying (-)
MaritalStatus_Single                           0.6003  -> TOWARD leaving (+)
JobLevel                                      -0.5695  -> TOWARD staying (-)
Age                                            0.5124  -> TOWARD leaving (+)


In [9]:
# Waterfall: True Negative (stayer)
fig = plt.figure(figsize=(12, 8))
shap.plots.waterfall(explanation[tn_idx], max_display=15, show=False)
plt.title(f'SHAP Waterfall — Correctly Predicted STAYER (test index {tn_idx})', fontsize=12)
plt.tight_layout()
wf_stayer_path = os.path.join(REPORTS, 'waterfall_stayer_example.png')
plt.savefig(wf_stayer_path, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'Saved: {os.path.abspath(wf_stayer_path)}  ({os.path.getsize(wf_stayer_path):,} bytes)')

# Print top driving SHAP values for this employee
emp_shap_tn = pd.Series(explanation[tn_idx].values, index=X.columns).sort_values(key=abs, ascending=False)
print(f'\nTop 8 SHAP drivers for STAYER (index {tn_idx}):')
print(f'{"Feature":<40} {"SHAP value":>12}  Direction')
print('-'*65)
for feat, val in emp_shap_tn.head(8).items():
    direction = '-> TOWARD leaving (+)' if val > 0 else '-> TOWARD staying (-)'
    print(f'{feat:<40} {val:>12.4f}  {direction}')

Saved: C:\Users\ASUS\Desktop\enterprise_hr_ai\reports\shap\waterfall_stayer_example.png  (175,911 bytes)

Top 8 SHAP drivers for STAYER (index 260):
Feature                                    SHAP value  Direction
-----------------------------------------------------------------
BusinessTravel_Travel_Rarely                  -0.6424  -> TOWARD staying (-)
OverTime                                      -0.4877  -> TOWARD staying (-)
TrainingTimesLastYear                         -0.3883  -> TOWARD staying (-)
BusinessTravel_Travel_Frequently              -0.3570  -> TOWARD staying (-)
YearsSinceLastPromotion                       -0.3553  -> TOWARD staying (-)
DistanceFromHome                              -0.2965  -> TOWARD staying (-)
JobSatisfaction                               -0.2875  -> TOWARD staying (-)
MaritalStatus_Single                          -0.2825  -> TOWARD staying (-)


---
## Step 8 · Cross-Check: SHAP vs LR Coefficients vs XGBoost Importances

In [10]:
print('=== CROSS-CHECK: SHAP GLOBAL IMPORTANCE vs STEP 6 LR COEFFICIENTS ===')
print()

# LR coefficient ranking (abs) from Step 6
lr_coef_top10 = [
    'OverTime', 'BusinessTravel_Travel_Frequently', 'JobRole_Laboratory Technician',
    'JobRole_Sales Representative', 'EducationField_Other', 'YearsSinceLastPromotion',
    'JobRole_Research Director', 'TotalWorkingYears', 'MaritalStatus_Single',
    'BusinessTravel_Travel_Rarely'
]

# XGBoost importance top 10 from Step 7
xgb_top10 = [
    'JobRole_Research Director', 'TotalWorkingYears', 'OverTime', 'JobLevel',
    'JobRole_Sales Executive', 'JobRole_Research Scientist', 'Department_Sales',
    'YearsWithCurrManager', 'StockOptionLevel', 'EnvironmentSatisfaction'
]

shap_top10 = mean_abs_shap.head(10).index.tolist()

print(f'{"Rank":<5} {"SHAP (this notebook)":<40} {"LR coeff (Step 6)":<40} {"XGB imp (Step 7)":<40}')
print('-'*130)
for i in range(10):
    s = shap_top10[i] if i < len(shap_top10) else ''
    c = lr_coef_top10[i] if i < len(lr_coef_top10) else ''
    x = xgb_top10[i] if i < len(xgb_top10) else ''
    agree_lr  = '✓' if s == c else ' '
    agree_xgb = '✓' if s == x else ' '
    print(f'{i+1:<5} {s:<40} {c:<40} {x:<40}  LR:{agree_lr} XGB:{agree_xgb}')

# Agreements
agree_lr_set  = set(shap_top10) & set(lr_coef_top10)
agree_xgb_set = set(shap_top10) & set(xgb_top10)
print()
print(f'Features in BOTH SHAP top-10 AND LR coeff top-10 ({len(agree_lr_set)}): {agree_lr_set}')
print(f'Features in BOTH SHAP top-10 AND XGB top-10 ({len(agree_xgb_set)}): {agree_xgb_set}')
print()
print('Cross-check interpretation:')
print('- SHAP on a LinearExplainer MUST closely mirror LR coefficients (phi_j = coef_j * (x_j - E[x_j]))')
print('  Strong agreement expected and confirms explainer is working correctly.')
print('- SHAP vs XGBoost: disagreement is EXPECTED and informative.')
print('  XGBoost captures non-linear threshold effects; LR/SHAP captures marginal linear contributions.')
print('  Where they AGREE (e.g. OverTime, TotalWorkingYears) -> robust, model-agnostic drivers.')
print('  Where they DISAGREE -> investigate whether the signal is linear or threshold-based.')

=== CROSS-CHECK: SHAP GLOBAL IMPORTANCE vs STEP 6 LR COEFFICIENTS ===

Rank  SHAP (this notebook)                     LR coeff (Step 6)                        XGB imp (Step 7)                        
----------------------------------------------------------------------------------------------------------------------------------
1     OverTime                                 OverTime                                 JobRole_Research Director                 LR:✓ XGB: 
2     YearsSinceLastPromotion                  BusinessTravel_Travel_Frequently         TotalWorkingYears                         LR:  XGB: 
3     TotalWorkingYears                        JobRole_Laboratory Technician            OverTime                                  LR:  XGB: 
4     BusinessTravel_Travel_Frequently         JobRole_Sales Representative             JobLevel                                  LR:  XGB: 
5     JobLevel                                 EducationField_Other                     JobRole_Sales Exe

---
## Step 9 · Plain-Language HR Summary

The following explanation is written for an HR manager audience — no data science jargon.
It will be reused verbatim in the Day 4 dashboard.

---

### What is driving employee attrition at this company?

Our AI model analyzed 1,470 employee records and identified five factors that are most strongly
linked to an employee deciding to leave:

1. **Working Overtime** is the single strongest warning sign. Employees who regularly work
   overtime are significantly more likely to leave — suggesting that sustained overwork is
   burning people out. The fix is workload review, not just salary review.

2. **Frequent Business Travel** is the second-strongest driver. Employees who travel frequently
   for work are much more at risk than those who never travel. Consider whether travel demands
   can be reduced through remote options or better trip scheduling.

3. **Years Without a Promotion** — employees who have gone a long time without advancing in
   their career, relative to how long they've been at the company, feel stuck. Career development
   conversations and promotion eligibility reviews matter here.

4. **Being Single (Marital Status)** is a demographic factor the model has found to correlate
   with higher attrition. Single employees have fewer financial and personal anchors that make
   switching jobs costly for them. This doesn't mean target single employees — it means ensure
   competitive packages and career paths for all, but especially early-career staff.

5. **Job Role — Laboratory Technicians and Sales Representatives** are the two job roles with
   the highest attrition risk. Both have highly portable skills and active external job markets.
   Targeted retention programs (pay benchmarking, role enrichment) for these two groups would
   have the highest return on investment.

> **Important note for HR managers:** The model does not tell you what to DO — it tells you
> WHERE to look. Each at-risk employee flagged by the system should be reviewed individually
> by their manager using these signals as a conversation guide, not as an automated action trigger.

---

In [11]:
print('=== PLAIN-LANGUAGE HR SUMMARY (for Day 4 dashboard) ===')
print()
print('TOP 5 SHAP DRIVERS IN HR LANGUAGE:')
top5 = mean_abs_shap.head(5)
hr_labels = {
    'OverTime': 'Employees who regularly work overtime are significantly more likely to leave. Review workload distribution.',
    'BusinessTravel_Travel_Frequently': 'Frequent business travel is a major burnout risk. Audit travel requirements and enable remote options.',
    'YearsSinceLastPromotion': 'Long stretches without promotion signal career stagnation. Schedule development conversations with long-tenured non-promoted staff.',
    'MaritalStatus_Single': 'Single employees have lower switching costs. Ensure competitive packages for early-career and single employees.',
    'JobRole_Laboratory Technician': 'Lab Technicians are high-risk due to portable, marketable skills. Benchmark pay and enrich roles proactively.',
    'JobRole_Sales Representative': 'Sales Reps have the highest market turnover rates. Evaluate commission structure, management quality, and career path.',
    'TotalWorkingYears': 'Employees earlier in their careers are more mobile. Invest in onboarding depth and early career mentoring.',
    'JobLevel': 'Lower job levels (junior staff) are more likely to leave. Clarify promotion timelines for this group.',
    'overall_satisfaction_score': 'Low satisfaction composite across job, environment, and relationships strongly predicts leaving. Run pulse surveys.',
    'income_per_year_at_company': 'Employees who earn little relative to their loyalty tenure feel undervalued. Conduct tenure-adjusted pay reviews.',
}
for rank, (feat, shap_val) in enumerate(top5.items(), 1):
    label = hr_labels.get(feat, f'Feature {feat!r} — review its relationship to attrition manually.')
    print(f'{rank}. [{feat}]  mean|SHAP|={shap_val:.4f}')
    print(f'   HR Insight: {label}')
    print()

=== PLAIN-LANGUAGE HR SUMMARY (for Day 4 dashboard) ===

TOP 5 SHAP DRIVERS IN HR LANGUAGE:
1. [OverTime]  mean|SHAP|=0.6558
   HR Insight: Employees who regularly work overtime are significantly more likely to leave. Review workload distribution.

2. [YearsSinceLastPromotion]  mean|SHAP|=0.5663
   HR Insight: Long stretches without promotion signal career stagnation. Schedule development conversations with long-tenured non-promoted staff.

3. [TotalWorkingYears]  mean|SHAP|=0.5573
   HR Insight: Employees earlier in their careers are more mobile. Invest in onboarding depth and early career mentoring.

4. [BusinessTravel_Travel_Frequently]  mean|SHAP|=0.5177
   HR Insight: Frequent business travel is a major burnout risk. Audit travel requirements and enable remote options.

5. [JobLevel]  mean|SHAP|=0.4585
   HR Insight: Lower job levels (junior staff) are more likely to leave. Clarify promotion timelines for this group.



---
## Step 10 · Output File Inventory

In [12]:
print('=== REPORTS/SHAP/ DIRECTORY ===')
for fname in sorted(os.listdir(REPORTS)):
    fpath = os.path.join(REPORTS, fname)
    if os.path.isfile(fpath):
        print(f'  {fname:<40}  {os.path.getsize(fpath):>10,} bytes')

print()
all_saved = [
    ('summary_beeswarm.png',        beeswarm_path),
    ('global_importance.png',        bar_path),
    ('waterfall_leaver_example.png', wf_leaver_path),
    ('waterfall_stayer_example.png', wf_stayer_path),
]
for name, path in all_saved:
    exists = os.path.exists(path) and os.path.getsize(path) > 0
    status = 'SAVED' if exists else 'MISSING'
    print(f'  [{status}] {name}')

=== REPORTS/SHAP/ DIRECTORY ===
  global_importance.png                        169,371 bytes
  summary_beeswarm.png                         204,292 bytes
  waterfall_leaver_example.png                 168,520 bytes
  waterfall_stayer_example.png                 175,911 bytes

  [SAVED] summary_beeswarm.png
  [SAVED] global_importance.png
  [SAVED] waterfall_leaver_example.png
  [SAVED] waterfall_stayer_example.png
